In [37]:
import sys
import numpy as np
import pymupdf
import ollama
import chromadb

print("Python:", sys.executable)
print("PyMuPDF:", pymupdf.__version__)
print("NumPy:", np.__version__)

Python: c:\Users\shwet\Desktop\GenAI\PROJECTS\LargeDocRAG\.venv\Scripts\python.exe
PyMuPDF: 1.28.2
NumPy: 2.5.3


In [38]:
doc = pymupdf.open("data/bookL.pdf")

print("Total pages:", len(doc))

Total pages: 513


In [39]:
# Inspect the first three pages, or fewer if the PDF is shorter
for page_index in range(min(3, len(doc))):
    text = doc[page_index].get_text()

    print(f"\n--- PDF page {page_index + 1} ---")
    print(f"Characters extracted: {len(text)}")
    print(text[:2000])


--- PDF page 1 ---
Characters extracted: 212
Mining
of
Massive
Datasets
Jure Leskovec
Stanford Univ.
Anand Rajaraman
Milliway Labs
Jeﬀrey D. Ullman
Stanford Univ.
Copyright c⃝2010, 2011, 2012, 2013, 2014 Anand Rajaraman, Jure Leskovec,
and Jeﬀrey D. Ullman


--- PDF page 2 ---
Characters extracted: 3
ii


--- PDF page 3 ---
Characters extracted: 1857
Preface
This book evolved from material developed over several years by Anand Raja-
raman and JeﬀUllman for a one-quarter course at Stanford.
The course
CS345A, titled “Web Mining,” was designed as an advanced graduate course,
although it has become accessible and interesting to advanced undergraduates.
When Jure Leskovec joined the Stanford faculty, we reorganized the material
considerably. He introduced a new course CS224W on network analysis and
added material to CS345A, which was renumbered CS246. The three authors
also introduced a large-scale data-mining project course, CS341. The book now
contains material taught in all three cours

In [40]:
#Extract all pages
pages = []

for page_index in range(len(doc)):
    text = doc[page_index].get_text()

    pages.append({
        "page_number": page_index + 1,
        "text": text
    })

print(f"Extracted {len(pages)} pages.")

Extracted 513 pages.


In [41]:
#Check empty pages
empty_pages = [
    page["page_number"]
    for page in pages
    if not page["text"].strip()
]

print("Total pages:", len(pages))
print("Pages with no extracted text:", len(empty_pages))
print("Their page numbers:", empty_pages)

Total pages: 513
Pages with no extracted text: 0
Their page numbers: []


In [42]:
#Creating Chunks
chunk_size = 1000
overlap = 200

chunks = []

for page in pages:
    text = page["text"]
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        if chunk_text.strip():
            chunks.append({
                "page_number": page["page_number"],
                "text": chunk_text
            })

        if end >= len(text):
            break

        start = end - overlap

print("Total chunks:", len(chunks))

Total chunks: 1604


In [43]:
#Inspecting 1st Chunk
print("First chunk's page:", chunks[0]["page_number"])
print(chunks[0]["text"])

First chunk's page: 1
Mining
of
Massive
Datasets
Jure Leskovec
Stanford Univ.
Anand Rajaraman
Milliway Labs
Jeﬀrey D. Ullman
Stanford Univ.
Copyright c⃝2010, 2011, 2012, 2013, 2014 Anand Rajaraman, Jure Leskovec,
and Jeﬀrey D. Ullman



In [44]:
#Checking Overlap
for i in range(len(chunks) - 1):
    first = chunks[i]
    second = chunks[i + 1]

    if first["page_number"] == second["page_number"]:
        print("PDF page:", first["page_number"])

        print("\nLast 200 characters of first chunk:")
        print(first["text"][-200:])

        print("\nFirst 200 characters of next chunk:")
        print(second["text"][:200])

        print(
            "\nOverlap matches:",
            first["text"][-200:] == second["text"][:200]
        )
        break

PDF page: 3

Last 200 characters of first chunk:
f data, that is, data so large
it does not ﬁt in main memory. Because of the emphasis on size, many of our
examples are about the Web or data derived from the Web. Further, the book
takes an algorithm

First 200 characters of next chunk:
f data, that is, data so large
it does not ﬁt in main memory. Because of the emphasis on size, many of our
examples are about the Web or data derived from the Web. Further, the book
takes an algorithm

Overlap matches: True


In [45]:
#Connect to Persistent Vector Database
chroma_client = chromadb.PersistentClient(
    path="vector_store"
)

collection = chroma_client.get_or_create_collection(
    name="bookl_chunks",
    metadata={"hnsw:space": "cosine"}
)

print("Existing records:", collection.count())

Existing records: 1604


In [46]:
#Generate Embeddings only when needed
expected_chunks = len(chunks)
stored_chunks = collection.count()

print("Expected chunks:", expected_chunks)
print("Stored chunks:", stored_chunks)

if stored_chunks == expected_chunks:
    print("All embeddings already exist. Skipping embedding generation.")

elif stored_chunks == 0:
    print("No embeddings found. Generating embeddings...")

    texts = [chunk["text"] for chunk in chunks]

    all_embeddings = []
    batch_size = 32

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = ollama.embed(
            model="embeddinggemma",
            input=batch
        )

        all_embeddings.extend(response["embeddings"])

        completed = min(start + batch_size, len(texts))
        print(f"Embedded {completed} of {len(texts)} chunks")

    print("Total embeddings:", len(all_embeddings))
    print("Values per embedding:", len(all_embeddings[0]))

    # Prepare ChromaDB records
    ids = [f"chunk-{i}" for i in range(len(chunks))]

    documents = [
        chunk["text"]
        for chunk in chunks
    ]

    metadatas = [
        {"page_number": chunk["page_number"]}
        for chunk in chunks
    ]

    # Store in batches
    batch_size = 100

    for start in range(0, len(chunks), batch_size):
        end = start + batch_size

        collection.upsert(
            ids=ids[start:end],
            documents=documents[start:end],
            embeddings=all_embeddings[start:end],
            metadatas=metadatas[start:end]
        )

    print("Embeddings stored:", collection.count())

else:
    raise RuntimeError(
        "Chroma contains a partial or mismatched index. "
        "Check the stored collection before continuing."
    )

Expected chunks: 1604
Stored chunks: 1604
All embeddings already exist. Skipping embedding generation.


In [47]:
#Retrieve embeddings from ChromaDB to verify
stored_data = collection.get(
    include=["embeddings"]
)

all_embeddings = stored_data["embeddings"]

print("Total embeddings:", len(all_embeddings))
print("Values per embedding:", len(all_embeddings[0]))

Total embeddings: 1604
Values per embedding: 768


In [1]:
import ollama

response = ollama.embed(
    model="embeddinggemma",
    input="What is the MapReduce programming model?"
)

embedding = response["embeddings"][0]

print("Embedding dimensions:", len(embedding))
print("First 10 values:", embedding[:10])

Embedding dimensions: 768
First 10 values: [-0.09476599, -0.029115869, 0.030197663, 0.004445703, 0.027578665, 0.05233326, -0.061625443, -0.04915502, 0.06530126, 0.027280437]


In [48]:
#Test one record from ChromaDB
sample_with_embeddings = collection.get(
    ids=["chunk-0"],
    include=["documents", "metadatas", "embeddings"]
)

print(
    "Embedding length:",
    len(sample_with_embeddings["embeddings"][0])
)

print(
    "First five values:",
    sample_with_embeddings["embeddings"][0][:5]
)

Embedding length: 768
First five values: [-0.07674275 -0.02356217  0.04328079  0.01111414 -0.03316089]


In [49]:
#Manual cosine similarity check
question = "What is the MapReduce programming model?"

question_response = ollama.embed(
    model="embeddinggemma",
    input=question
)

question_embedding = np.array(
    question_response["embeddings"][0]
)

document_embeddings = np.array(all_embeddings)

# Cosine similarity between question and every document chunk
question_norm = np.linalg.norm(question_embedding)

document_norms = np.linalg.norm(
    document_embeddings,
    axis=1
)

similarities = (
    document_embeddings @ question_embedding
    / (document_norms * question_norm)
)

top_k = 5

top_indices = np.argsort(similarities)[-top_k:][::-1]

for rank, index in enumerate(top_indices, start=1):

    print(f"\n--- Result {rank} ---")
    print(
        "Similarity:",
        round(float(similarities[index]), 4)
    )

    print(
        "PDF page:",
        chunks[index]["page_number"]
    )

    print(
        chunks[index]["text"][:500]
    )


--- Result 1 ---
Similarity: 0.586
PDF page: 46
tion
Let us now consider in more detail how a program using MapReduce is executed.
Figure 2.3 oﬀers an outline of how processes, tasks, and ﬁles interact. Taking
advantage of a library provided by a MapReduce system such as Hadoop, the
user program forks a Master controller process and some number of Worker
processes at diﬀerent compute nodes. Normally, a Worker handles either Map
tasks (a Map worker) or Reduce tasks (a Reduce worker), but not both.
The Master has many responsibilities. One is t

--- Result 2 ---
Similarity: 0.5773
PDF page: 87
 to measure
communication by the replication rate, which is the communication per
input. Also, the reducer size is the maximum number of inputs associated
with any reducer. For many problems, it is possible to derive a lower
bound on replication rate as a function of the reducer size.
Representing Problems as Graphs: It is possible to represent many prob-
lems that are amenable to MapReduce comput

In [50]:
# Retrieve similar chunks using ChromaDB

top_k = 5

chroma_results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=top_k,
    include=["documents", "metadatas", "distances"]
)

for rank, (document, metadata, distance) in enumerate(
    zip(
        chroma_results["documents"][0],
        chroma_results["metadatas"][0],
        chroma_results["distances"][0]
    ),
    start=1
):
    print(f"\n--- ChromaDB result {rank} ---")
    print("Cosine distance:", round(float(distance), 4))
    print("PDF page:", metadata["page_number"])
    print(document[:500])


--- ChromaDB result 1 ---
Cosine distance: 0.414
PDF page: 46
tion
Let us now consider in more detail how a program using MapReduce is executed.
Figure 2.3 oﬀers an outline of how processes, tasks, and ﬁles interact. Taking
advantage of a library provided by a MapReduce system such as Hadoop, the
user program forks a Master controller process and some number of Worker
processes at diﬀerent compute nodes. Normally, a Worker handles either Map
tasks (a Map worker) or Reduce tasks (a Reduce worker), but not both.
The Master has many responsibilities. One is t

--- ChromaDB result 2 ---
Cosine distance: 0.4227
PDF page: 87
 to measure
communication by the replication rate, which is the communication per
input. Also, the reducer size is the maximum number of inputs associated
with any reducer. For many problems, it is possible to derive a lower
bound on replication rate as a function of the reducer size.
Representing Problems as Graphs: It is possible to represent many prob-
lems that are 

In [51]:
#Get User's input
question = input("Ask a question about the book: ")

print("Your question:", question)

question_response = ollama.embed(
    model="embeddinggemma",
    input=question
)

question_embedding = question_response["embeddings"][0]

Your question: What is the Market-basket model? Explain in 3 points


In [52]:
#Query ChromaDB for similar chunks
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

In [53]:
for i in range(3):
    print(f"\n--- Result {i + 1} ---")
    print("Chunk ID:", results["ids"][0][i])
    print("Distance:", results["distances"][0][i])
    print("Page:", results["metadatas"][0][i]["page_number"])
    print("Text:")
    print(results["documents"][0][i][:1000])


--- Result 1 ---
Chunk ID: chunk-718
Distance: 0.49412792921066284
Page: 225
Text:
6.1. THE MARKET-BASKET MODEL
207
appears in only one of the eight baskets. This interest, while positive, is close
to 0 and therefore indicates the association rule is not very “interesting.”
✷
6.1.4
Finding Association Rules with High Conﬁdence
Identifying useful association rules is not much harder than ﬁnding frequent
itemsets.
We shall take up the problem of ﬁnding frequent itemsets in the
balance of this chapter, but for the moment, assume it is possible to ﬁnd those
frequent itemsets whose support is at or above a support threshold s.
If we are looking for association rules I →j that apply to a reasonable
fraction of the baskets, then the support of I must be reasonably high.
In
practice, such as for marketing in brick-and-mortar stores, “reasonably high”
is often around 1% of the baskets. We also want the conﬁdence of the rule to
be reasonably high, perhaps 50%, or else the rule has little practi

In [54]:
# Build context from the retrieved ChromaDB results

retrieved_documents = results["documents"][0]
retrieved_metadata = results["metadatas"][0]

context_parts = []

for document, metadata in zip(
    retrieved_documents,
    retrieved_metadata
):
    page_number = metadata["page_number"]

    context_parts.append(
        f"[PDF page {page_number}]\n{document}"
    )

context = "\n\n---\n\n".join(context_parts)

print(context[:2000])

[PDF page 225]
6.1. THE MARKET-BASKET MODEL
207
appears in only one of the eight baskets. This interest, while positive, is close
to 0 and therefore indicates the association rule is not very “interesting.”
✷
6.1.4
Finding Association Rules with High Conﬁdence
Identifying useful association rules is not much harder than ﬁnding frequent
itemsets.
We shall take up the problem of ﬁnding frequent itemsets in the
balance of this chapter, but for the moment, assume it is possible to ﬁnd those
frequent itemsets whose support is at or above a support threshold s.
If we are looking for association rules I →j that apply to a reasonable
fraction of the baskets, then the support of I must be reasonably high.
In
practice, such as for marketing in brick-and-mortar stores, “reasonably high”
is often around 1% of the baskets. We also want the conﬁdence of the rule to
be reasonably high, perhaps 50%, or else the rule has little practical eﬀect. As
a result, the set I ∪{j} will also have fairly high sup

In [55]:
#Sending context to Ollama for answer generation
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "system",
            "content": (
                "You answer questions using only the supplied PDF context. "
                "If the answer is not supported by the context, say that "
                "the PDF context does not provide enough information. "
                "Include the relevant PDF page numbers in your answer."
            )
        },
        {
            "role": "user",
            "content": f"""
Question:
{question}

PDF context:
{context}

Answer the question using the PDF context.
"""
        }
    ]
)

print(response["message"]["content"])

Here are 3 points explaining the Market-basket model:

1. **Definition**: The Market-basket model describes a many-many relationship between two kinds of objects: items and baskets (also called transactions). Each basket is a set of items, and the number of items in a basket is assumed to be small compared to the total number of items. (PDF page 220, 6.1)

2. **Assumptions**: The data is assumed to be represented in a file consisting of a sequence of baskets, where each basket is of type "set of items". The number of baskets is assumed to be very large, bigger than what can fit in main memory. (PDF page 220, 6.1)

3. **Frequent Itemsets**: A set of items is considered "frequent" if it appears in many baskets, and is defined by a support threshold s. For example, if the support threshold is 10,000 and there are one million items, a set of items is frequent if it appears at least 10,000 times. (PDF page 235, Exercise 6.2.7, assumptions 1 and 2)
